In [4]:
import pandas as pd
import numpy as np
import re
import warnings
import gc
import joblib
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
import lightgbm as lgb
import xgboost as xgb

try:
    import catboost as cb
    CATBOOST_AVAILABLE = True
except:
    CATBOOST_AVAILABLE = False

warnings.filterwarnings('ignore')

def remove_emojis(text):
    if pd.isna(text): return ''
    emoji_pattern = re.compile(
        "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF\U00002500-\U00002BEF\U00002702-\U000027B0"
        "\U000024C2-\U0001F251\U0001F900-\U0001F9FF\U0001FA00-\U0001FAFF"
        "\U00002600-\U000026FF\U00002700-\U000027BF]+", flags=re.UNICODE)
    return emoji_pattern.sub('', str(text)).strip()

def extract_item_pack_quantity(text):
    if not isinstance(text, str): return 1
    text_lower = text.lower()
    match = re.search(r'(\d+)\s*(?:pack|pk|count|ct|piece|units?)\s+of\s+(\d+)|(?:pack|set)\s+of\s+(\d+)|(\d+)\s*[-\s]?(?:pack|count|ct|piece|units?)\b|quantity[:\s]+(\d+)', text_lower)
    if match:
        for g in match.groups():
            if g:
                try:
                    qty = int(g)
                    if 1 <= qty <= 1000: return qty
                except: pass
    return 1

def extract_brand(text):
    if not isinstance(text, str) or len(text.strip()) == 0: return 'unknown'
    text = text.strip()
    match = re.match(r'^([A-Z][A-Za-z0-9&\s]+?)(?:,|-|\s+\d)', text)
    if match:
        return ' '.join(match.group(1).strip().lower().split()[:2])
    words = text.split()
    return words[0].lower() if words and len(words[0]) > 1 else 'generic'

def detect_product_category(text):
    if not isinstance(text, str): return 'other'
    text_lower = text.lower()
    if any(k in text_lower for k in ['electronic', 'battery', 'charger', 'cable']): return 'electronics'
    if any(k in text_lower for k in ['food', 'snack', 'candy', 'chocolate', 'chip']): return 'food'
    if any(k in text_lower for k in ['drink', 'beverage', 'water', 'juice', 'soda']): return 'beverage'
    if any(k in text_lower for k in ['vitamin', 'supplement', 'protein', 'medicine']): return 'health'
    if any(k in text_lower for k in ['beauty', 'cosmetic', 'shampoo', 'soap', 'lotion']): return 'beauty'
    if any(k in text_lower for k in ['paper', 'towel', 'cleaning', 'detergent']): return 'home'
    if any(k in text_lower for k in ['pet', 'dog', 'cat', 'animal']): return 'pet'
    if any(k in text_lower for k in ['baby', 'infant', 'diaper', 'formula']): return 'baby'
    return 'other'

def extract_fields_vectorized(df):
    item_names = df['catalog_content'].str.extract(r'Item Name:\s*(.+?)(?=\n|Bullet Point|Product Description|Value:|$)', flags=re.IGNORECASE)[0].fillna('')
    values = df['catalog_content'].str.extract(r'Value:\s*([\d.]+)', flags=re.IGNORECASE)[0]
    units = df['catalog_content'].str.extract(r'Unit:\s*([^\n]+)', flags=re.IGNORECASE)[0]
    
    df['item_name'] = item_names
    df['value'] = pd.to_numeric(values, errors='coerce')
    df['unit'] = units
    
    df['catalog_content_cleaned'] = df['catalog_content'].str.replace(
        r'Item Name:.*?(?=Bullet Point|Product Description|Value:|$)|Value:.*?(?=Unit:|$)|Unit:.*?$', 
        '', regex=True, flags=re.IGNORECASE).str.strip()
    return df

def standardize_units_vectorized(df):
    weight_map = {'ounce': 28.3495, 'oz': 28.3495, 'pound': 453.592, 'lb': 453.592, 
                  'kilogram': 1000, 'kg': 1000, 'gram': 1, 'g': 1}
    volume_map = {'fl oz': 29.5735, 'fluid ounce': 29.5735, 'liter': 1000, 'l': 1000, 
                  'ml': 1, 'milliliter': 1}
    
    df['unit_lower'] = df['unit'].str.lower().str.strip()
    df['standardized_value'] = df['value']
    df['standardized_unit'] = df['unit']
    
    for unit, factor in weight_map.items():
        mask = df['unit_lower'] == unit
        df.loc[mask, 'standardized_value'] = df.loc[mask, 'value'] * factor
        df.loc[mask, 'standardized_unit'] = 'grams'
    
    for unit, factor in volume_map.items():
        mask = df['unit_lower'] == unit
        df.loc[mask, 'standardized_value'] = df.loc[mask, 'value'] * factor
        df.loc[mask, 'standardized_unit'] = 'ml'
    
    count_mask = df['unit_lower'].isin(['count', 'ct'])
    df.loc[count_mask, 'standardized_unit'] = 'count'
    
    df.drop('unit_lower', axis=1, inplace=True)
    return df

def extract_features_vectorized(df):
    text = df['catalog_content_cleaned'].fillna('').astype(str).str.lower()
    
    df['gluten_free'] = text.str.contains(r'gluten[- ]?free', regex=True).astype(int)
    df['organic'] = text.str.contains(r'organic|usda', regex=True).astype(int)
    df['non_gmo'] = text.str.contains(r'non[- ]?gmo', regex=True).astype(int)
    df['vegan'] = text.str.contains(r'vegan|plant[- ]?based', regex=True).astype(int)
    df['dairy_free'] = text.str.contains(r'dairy[- ]?free', regex=True).astype(int)
    df['sugar_free'] = text.str.contains(r'sugar[- ]?free|zero sugar', regex=True).astype(int)
    df['natural'] = text.str.contains(r'natural|all-natural', regex=True).astype(int)
    df['is_premium'] = text.str.contains(r'premium|luxury|deluxe|gourmet', regex=True).astype(int)
    df['is_travel_size'] = text.str.contains(r'travel[- ]?size|mini|portable', regex=True).astype(int)
    df['is_bulk'] = text.str.contains(r'bulk|wholesale|economy', regex=True).astype(int)
    df['bullet_point_count'] = text.str.findall(r'bullet point\s*\d*:').str.len().fillna(0).astype(int)
    
    return df

def create_advanced_features(df):
    df['value_per_unit'] = df['standardized_value'] / (df['item_pack_quantity'] + 1)
    df['total_volume'] = df['standardized_value'] * df['item_pack_quantity']
    df['value_per_unit_log'] = np.log1p(df['value_per_unit'])
    df['total_volume_log'] = np.log1p(df['total_volume'])
    df['standardized_value_log'] = np.log1p(df['standardized_value'])
    df['standardized_value_sqrt'] = np.sqrt(df['standardized_value'])
    
    df['ipq_squared'] = df['item_pack_quantity'] ** 2
    df['ipq_sqrt'] = np.sqrt(df['item_pack_quantity'])
    df['ipq_log'] = np.log1p(df['item_pack_quantity'])
    df['ipq_cubed'] = df['item_pack_quantity'] ** 3
    df['is_common_pack'] = df['item_pack_quantity'].isin([1,2,4,6,8,12,24]).astype(int)
    df['is_bulk_pack'] = (df['item_pack_quantity'] >= 12).astype(int)
    df['is_single_item'] = (df['item_pack_quantity'] == 1).astype(int)
    df['is_double_pack'] = (df['item_pack_quantity'] == 2).astype(int)
    
    brand_counts = df['brand'].value_counts()
    df['brand_frequency'] = df['brand'].map(brand_counts)
    df['brand_freq_log'] = np.log1p(df['brand_frequency'])
    df['is_rare_brand'] = (df['brand_frequency'] <= 3).astype(int)
    df['is_popular_brand'] = (df['brand_frequency'] >= 20).astype(int)
    df['is_medium_brand'] = ((df['brand_frequency'] > 3) & (df['brand_frequency'] < 20)).astype(int)
    
    df['item_words'] = df['item_name'].fillna('').str.split().str.len()
    df['catalog_words'] = df['catalog_content_cleaned'].fillna('').str.split().str.len()
    df['item_chars'] = df['item_name'].fillna('').str.len()
    df['catalog_chars'] = df['catalog_content_cleaned'].fillna('').str.len()
    df['avg_word_length_item'] = df['item_chars'] / (df['item_words'] + 1)
    df['avg_word_length_catalog'] = df['catalog_chars'] / (df['catalog_words'] + 1)
    
    df['value_x_ipq'] = df['standardized_value'] * df['item_pack_quantity']
    df['value_x_brand_freq'] = df['standardized_value'] * df['brand_freq_log']
    df['value_log_x_ipq_log'] = df['standardized_value_log'] * df['ipq_log']
    df['value_sqrt_x_ipq_sqrt'] = df['standardized_value_sqrt'] * df['ipq_sqrt']
    
    df['is_likely_expensive'] = ((df['standardized_value'] > df['standardized_value'].quantile(0.75)) | 
                                  (df['item_pack_quantity'] >= 6)).astype(int)
    df['is_likely_cheap'] = ((df['standardized_value'] < df['standardized_value'].quantile(0.25)) & 
                             (df['item_pack_quantity'] == 1)).astype(int)
    
    return df

def preprocess_data(df, is_train=True, artifacts=None, allow_drop=False):
    if artifacts is None: artifacts = {}
    
    df['catalog_content'] = df['catalog_content'].apply(remove_emojis)
    df = extract_fields_vectorized(df)
    df['item_pack_quantity'] = df['catalog_content'].apply(extract_item_pack_quantity)
    df['brand'] = df['item_name'].apply(extract_brand)
    
    if is_train:
        le = LabelEncoder()
        df['brand_encoded'] = le.fit_transform(df['brand'].astype(str))
        artifacts['brand_encoder'] = le
    else:
        le = artifacts['brand_encoder']
        brand_map = {label: idx for idx, label in enumerate(le.classes_)}
        df['brand_encoded'] = df['brand'].map(brand_map).fillna(-1).astype(int)
    
    df['product_category'] = df['catalog_content'].apply(detect_product_category)
    df = standardize_units_vectorized(df)
    
    if allow_drop:
        df = df.dropna(subset=['standardized_unit', 'standardized_value'])
        df = df[df['standardized_unit'].isin(['grams', 'ml', 'count'])]
    else:
        df['standardized_unit'] = df['standardized_unit'].fillna('count')
        df['standardized_value'] = df['standardized_value'].fillna(0)
    
    df = pd.get_dummies(df, columns=['standardized_unit', 'product_category'], 
                        prefix=['unit', 'cat'], dtype=int, dummy_na=True)
    
    if is_train:
        scaler = MinMaxScaler()
        df['value_scaled'] = scaler.fit_transform(df[['standardized_value']].fillna(0))
        artifacts['value_scaler'] = scaler
    else:
        df['value_scaled'] = artifacts['value_scaler'].transform(df[['standardized_value']].fillna(0))
    
    df = extract_features_vectorized(df)
    df = create_advanced_features(df)
    
    if is_train and 'price' in df.columns:
        brand_stats = df.groupby('brand')['price'].agg(['mean', 'median', 'std', 'count']).reset_index()
        brand_stats.columns = ['brand', 'brand_price_mean', 'brand_price_median', 'brand_price_std', 'brand_price_count']
        
        global_mean = df['price'].mean()
        min_samples = 5
        brand_stats['brand_price_mean_smooth'] = (
            (brand_stats['brand_price_mean'] * brand_stats['brand_price_count'] + global_mean * min_samples) / 
            (brand_stats['brand_price_count'] + min_samples)
        )
        
        artifacts['brand_stats'] = brand_stats
        df = df.merge(brand_stats, on='brand', how='left')
        df['brand_price_mean_smooth'] = df['brand_price_mean_smooth'].fillna(global_mean)
        df['brand_price_std'] = df['brand_price_std'].fillna(df['price'].std())
    elif not is_train:
        brand_stats = artifacts['brand_stats']
        df = df.merge(brand_stats, on='brand', how='left')
        df['brand_price_mean_smooth'] = df['brand_price_mean_smooth'].fillna(artifacts['global_mean'])
        df['brand_price_std'] = df['brand_price_std'].fillna(artifacts['global_std'])
    
    if is_train and 'price' in df.columns:
        artifacts['global_mean'] = df['price'].mean()
        artifacts['global_std'] = df['price'].std()
    
    df['item_clean'] = df['item_name'].fillna('').str.replace(r'[^\w\s]', ' ', regex=True).str.split().str.join(' ')
    df['catalog_clean'] = df['catalog_content_cleaned'].str.replace(r'Bullet Point\s*\d*:\s*', '', regex=True, flags=re.IGNORECASE)
    
    combined_text = df['item_clean'].str.repeat(3) + ' ' + df['catalog_clean']
    
    if is_train:
        tfidf = TfidfVectorizer(max_features=120, stop_words='english', 
                                ngram_range=(1, 2), min_df=2, max_df=0.85,
                                sublinear_tf=True)
        tfidf_matrix = tfidf.fit_transform(combined_text)
        artifacts['tfidf'] = tfidf
        
        svd = TruncatedSVD(n_components=40, random_state=42)
        tfidf_reduced = svd.fit_transform(tfidf_matrix)
        artifacts['tfidf_svd'] = svd
    else:
        tfidf_matrix = artifacts['tfidf'].transform(combined_text)
        tfidf_reduced = artifacts['tfidf_svd'].transform(tfidf_matrix)
    
    tfidf_df = pd.DataFrame(tfidf_reduced, columns=[f'tfidf_{i}' for i in range(40)], index=df.index)
    df = pd.concat([df, tfidf_df], axis=1)
    
    return df, artifacts

class TargetTransformer:
    def __init__(self):
        self.lower, self.upper = None, None
    
    def fit(self, y):
        self.lower = np.percentile(y, 0.5)
        self.upper = np.percentile(y, 99.5)
        return self
    
    def transform(self, y):
        clipped = np.clip(y, self.lower, self.upper)
        return np.log1p(clipped)
    
    def inverse_transform(self, y_log):
        predictions = np.expm1(y_log)
        predictions = np.clip(predictions, 0.5, 500)
        return predictions

def smape(y_true, y_pred):
    num = np.abs(y_pred - y_true)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    ratio = np.zeros_like(num)
    ratio[mask] = num[mask] / denom[mask]
    return 100 * np.mean(ratio)

class OptimizedEnsemble:
    def __init__(self):
        self.lgb_model = None
        self.xgb_model = None
        self.cb_model = None
        self.feature_cols = None
        self.weights = {'lgb': 0.50, 'xgb': 0.35, 'cb': 0.15}
    
    def get_features(self, df):
        exclude = ['sample_id', 'image_link', 'price', 'item_name', 'catalog_content',
                   'value', 'unit', 'catalog_content_cleaned', 'brand',
                   'item_clean', 'catalog_clean']
        cols = [c for c in df.columns if c not in exclude and c in df.columns]
        numeric_cols = [col for col in cols if df[col].dtype in ['int64','float64','int32','float32','uint8','int16','int8']]
        return numeric_cols
    
    def train(self, df, y, target_transformer):
        self.feature_cols = self.get_features(df)
        X = df[self.feature_cols].copy()
        
        for col in X.columns:
            if X[col].dtype == 'object':
                X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)
        
        X = X.replace([np.inf, -np.inf], 0).fillna(0).values
        
        if isinstance(y, pd.Series):
            y = y.values
        
        self.lgb_model = lgb.LGBMRegressor(
            objective='regression', metric='rmse', n_estimators=2000, learning_rate=0.03,
            num_leaves=50, max_depth=9, min_child_samples=20, feature_fraction=0.7,
            bagging_fraction=0.7, bagging_freq=5, reg_alpha=1.0, reg_lambda=2.0,
            min_gain_to_split=0.01, random_state=42, n_jobs=-1, verbose=-1
        )
        self.lgb_model.fit(X, y)
        
        self.xgb_model = xgb.XGBRegressor(
            objective='reg:squarederror', n_estimators=2000, learning_rate=0.03,
            max_depth=8, min_child_weight=4, subsample=0.7, colsample_bytree=0.7,
            gamma=0.3, reg_alpha=1.0, reg_lambda=2.0, random_state=42,
            n_jobs=-1, tree_method='hist', verbosity=0
        )
        self.xgb_model.fit(X, y)
        
        if CATBOOST_AVAILABLE:
            self.cb_model = cb.CatBoostRegressor(
                iterations=1500, learning_rate=0.04, depth=8, l2_leaf_reg=8,
                random_seed=42, verbose=0
            )
            self.cb_model.fit(X, y)
        
        gc.collect()
    
    def predict(self, df):
        X = df[self.feature_cols].copy()
        for col in X.columns:
            if X[col].dtype == 'object':
                X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)
        X = X.replace([np.inf, -np.inf], 0).fillna(0).values
        
        lgb_pred = self.lgb_model.predict(X)
        xgb_pred = self.xgb_model.predict(X)
        
        pred = self.weights['lgb'] * lgb_pred + self.weights['xgb'] * xgb_pred
        
        if CATBOOST_AVAILABLE and self.cb_model:
            cb_pred = self.cb_model.predict(X)
            pred += self.weights['cb'] * cb_pred
        
        return pred

print("Loading training data...")
train = pd.read_csv('train.csv')
train_processed, artifacts = preprocess_data(train, is_train=True, allow_drop=True)

train_data, val_data = train_test_split(train_processed, test_size=0.2, random_state=42)

target_transformer = TargetTransformer()
y_train = target_transformer.fit(train_data['price']).transform(train_data['price'])

print("Training ensemble...")
ensemble = OptimizedEnsemble()
ensemble.train(train_data, y_train, target_transformer)

val_pred_log = ensemble.predict(val_data)
val_pred = target_transformer.inverse_transform(val_pred_log)

val_smape = smape(val_data['price'].values, val_pred)
print(f"Validation SMAPE: {val_smape:.4f}")

print("Saving models...")
joblib.dump(artifacts, 'artifacts.pkl')
joblib.dump(ensemble, 'ensemble.pkl')
joblib.dump(target_transformer, 'target_transformer.pkl')

print("\nLoading test data...")
test = pd.read_csv('test.csv')
original_test_ids = test['sample_id'].copy()

test_processed, _ = preprocess_data(test, is_train=False, artifacts=artifacts, allow_drop=False)

print("Generating predictions...")
predictions_log = ensemble.predict(test_processed)
predictions = target_transformer.inverse_transform(predictions_log)

output_df = pd.DataFrame({
    'sample_id': original_test_ids.values,
    'price': predictions
})

output_df.to_csv('test_out.csv', index=False)
print(f"Total predictions: {len(output_df)}")
print(output_df.head(10))

Loading training data...
Training ensemble...
Validation SMAPE: 35.1595
Saving models...

Loading test data...
Generating predictions...
Total predictions: 75000
   sample_id      price
0     100179  13.614161
1     245611  14.286082
2     146263   2.674408
3      95658   6.069605
4      36806  27.303822
5     148239   5.570966
6      92659   2.485701
7       3780   3.172514
8     196940   3.383187
9      20472   6.564220
